# LarNO — Quick Inference Demo

> **Large-scale urban flood modeling and zero-shot high-resolution generalization with LarNO**  
> *Journal of Hydrology (Under Review)*

This notebook runs **inference on the Futian (region1_20m) test set** using pre-trained LarNO weights.  
No training required — the full pipeline takes **~15 minutes** on a free Colab T4 GPU.

**What you will get:**
- Per-event metrics: R², MAE, MSE, RMSE, PeakR², CSI
- Animated GIF comparison: MIKE+ hydraulic solver (reference) vs LarNO prediction
- Side-by-side flood depth maps at each 5-minute time step

---
**Steps overview:**
1. Clone repo & install dependencies
2. Download pre-trained weights (Google Drive, ~50 MB)
3. Upload / download the dataset
4. Run inference
5. Visualise results

## Step 1 — Clone repo & install dependencies

> Runtime → Change runtime type → **T4 GPU** (free tier is fine)

In [ ]:
import os

# Always reset to /content first to avoid stale cwd errors after runtime restarts
os.chdir('/content')

# Clone the repository
if not os.path.exists('/content/LarNO'):
    !git clone https://github.com/holmescao/LarNO /content/LarNO
else:
    print('Repository already cloned.')

%cd /content/LarNO/code/urbanflood_larfno

In [ ]:
# Install the local package (neuralop) and all dependencies
!pip install -e . --no-deps -q
!pip install \
    tensorly tensorly-torch "torch-harmonics==0.7.3" \
    ruamel-yaml configmypy opt-einsum h5py zarr matplotlib \
    "numpy>=1.25" pandas tqdm scipy opencv-python openpyxl torchmetrics \
    -q

# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Download pre-trained weights

The pre-trained Futian (region1_20m) checkpoint is hosted on Google Drive.  
We use `gdown` to download it directly into the Colab instance.

In [ ]:
import os

os.chdir('/content')  # reset cwd in case of stale session state

!pip install gdown -q

REPO_ROOT  = '/content/LarNO'
EXP_ROOT   = f'{REPO_ROOT}/exp'
os.makedirs(EXP_ROOT, exist_ok=True)

WEIGHTS_ZIP    = '/content/exp_weights.zip'
GDRIVE_FILE_ID = '1ITPoTWQkm5v9kdZT9fqza2Xd4a6Lc-0t'  # Google Drive file ID from README

if not os.path.exists(WEIGHTS_ZIP):
    print('Downloading pre-trained weights from Google Drive ...')
    !gdown {GDRIVE_FILE_ID} -O {WEIGHTS_ZIP}
else:
    print('Weights zip already downloaded.')

# The zip contains exp/<expr_id>/weights/... so unzip to REPO_ROOT (not EXP_ROOT)
# to avoid the double-nesting exp/exp/... problem
!unzip -q -o {WEIGHTS_ZIP} -d {REPO_ROOT}
print('Contents of exp/:')
!ls {EXP_ROOT}

In [ ]:
import os

# Auto-detect the experiment folder name from the unzipped weights
exp_dirs = [d for d in os.listdir(EXP_ROOT)
            if os.path.isdir(os.path.join(EXP_ROOT, d))]
if not exp_dirs:
    raise RuntimeError('No experiment directory found under exp/. Check the zip structure.')

EXPR_ID = sorted(exp_dirs)[-1]  # take the latest
print(f'Using experiment: {EXPR_ID}')

# Verify checkpoint exists
weights_root = os.path.join(EXP_ROOT, EXPR_ID, 'weights')
if os.path.exists(weights_root):
    print('Checkpoint found:')
    for root, dirs, files in os.walk(weights_root):
        for f in files:
            print(' ', os.path.join(root, f))
else:
    print(f'WARNING: weights/ directory not found at {weights_root}')
    print('Please check the zip structure and unzip manually if needed.')

## Step 3 — Download the dataset

The benchmark dataset (region1_20m, Futian district Shenzhen) is downloaded automatically from Google Drive.

Expected structure after download:
```
/content/LarNO/benchmark/urbanflood/
    flood/region1_20m/<event_name>/
        rainfall.npy   (T=72, H=400, W=560)
        h.npy          (T=72, H=400, W=560)
    geodata/region1_20m/
        dem.npy        (H=400, W=560)
```

**Dataset page:** [https://holmescao.github.io/datasets/LarNO](https://holmescao.github.io/datasets/LarNO)

In [ ]:
import os

os.chdir('/content')  # reset cwd in case of stale session state

BENCHMARK_ROOT = '/content/LarNO/benchmark'

# ── Download benchmark dataset from Google Drive ─────────────────────────────
# Dataset page: https://holmescao.github.io/datasets/LarNO
DATASET_GDRIVE_ID = '1bw97nT9kA1qM8lCuzhzSU3X1KBkJ_Kdw'
DATASET_ZIP = '/content/benchmark.zip'

if not os.path.exists(DATASET_ZIP):
    print('Downloading benchmark dataset from Google Drive ...')
    !gdown {DATASET_GDRIVE_ID} -O {DATASET_ZIP}
else:
    print('Dataset zip already downloaded.')

# The zip contains benchmark/urbanflood/... so unzip to /content/LarNO/
!unzip -q -o {DATASET_ZIP} -d /content/LarNO/
print('Dataset unzipped.')

# ── Verify structure ──────────────────────────────────────────────────────────
region1_dir = os.path.join(BENCHMARK_ROOT, 'urbanflood/flood/region1_20m')
if os.path.exists(region1_dir):
    events = os.listdir(region1_dir)
    print(f'Found {len(events)} events in region1_20m:')
    for e in sorted(events)[:5]:
        print(' ', e)
    if len(events) > 5:
        print(f'  ... and {len(events) - 5} more')
else:
    print(f'WARNING: {region1_dir} not found. Please check the zip structure.')
    print('Expected: benchmark/urbanflood/flood/region1_20m/<event_name>/')
    !find /content/LarNO/benchmark -maxdepth 4 -type d 2>/dev/null | head -20

## Step 4 — Run inference

We call `test.py` with the Quick Test config (`urbanflood_config_2d.yaml`).  
This config already has the pre-trained weight path embedded — no manual editing needed.

Expected runtime: **~8 minutes** on T4 GPU for 16 test events.

In [ ]:
# Verify the config points to the correct paths
import yaml

config_path = '/content/LarNO/code/urbanflood_larfno/configs/urbanflood_config_2d.yaml'
with open(config_path, 'r') as f:
    cfg_raw = yaml.safe_load(f)

# The YAML has a 'default:' root key
cfg = cfg_raw.get('default', cfg_raw)

print('=== Key config values ===')
print(f"hidden_channels : {cfg['tfno2d']['hidden_channels']}  (must be 32)")
print(f"train_location  : {cfg['data']['train_location']}")
print(f"test_list       : {cfg['data']['test_list']}")
print(f"eval.locations  : {cfg['eval']['locations']}")
print(f"data_root       : {cfg['data']['data_root']}")
print(f"exp_root        : {cfg['data']['exp_root']}")
print(f"pretrained_dir  : {cfg['finetune']['pretrained_dir']}")

In [ ]:
# Patch data_root and exp_root to Colab absolute paths (overrides YAML relative paths)
import subprocess, sys

os.chdir('/content/LarNO/code/urbanflood_larfno')

cmd = [
    sys.executable, 'test.py',
    '--config', 'urbanflood_config_2d.yaml',
    '--expr_id', EXPR_ID,
    '--data_root', '/content/LarNO/benchmark/urbanflood',
    '--exp_root',  '/content/LarNO/exp',
    '--device', '0',
]

print('Running:', ' '.join(cmd))
print('This will take ~8 minutes on T4 GPU ...')

result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('ERROR: test.py exited with non-zero code', result.returncode)

## Step 5 — Visualise results

In [ ]:
import os
import glob
import pandas as pd

# Find the most recently created experiment output
exp_dirs = sorted(os.listdir('/content/LarNO/exp'))
# The inference run creates a NEW timestamped dir; it is the latest one
latest_exp = exp_dirs[-1]
print(f'Experiment output dir: {latest_exp}')

# Load metrics Excel
xlsx_files = glob.glob(f'/content/LarNO/exp/{latest_exp}/test_metrics/region1_20m/*.xlsx')
if xlsx_files:
    df = pd.read_excel(xlsx_files[0])
    print('\n=== Test Metrics (region1_20m) ===')
    display(df)
else:
    print('No metrics Excel found. Check that inference completed successfully.')

In [ ]:
import glob
from IPython.display import Image, display

# Find and display one of the output GIFs
gif_files = sorted(glob.glob(
    f'/content/LarNO/exp/{latest_exp}/visualization/region1_20m/**/*.gif',
    recursive=True
))

if gif_files:
    print(f'Found {len(gif_files)} GIF(s). Showing first one:')
    print(gif_files[0])
    display(Image(filename=gif_files[0]))
else:
    print('No GIF found. Check visualization directory.')
    vis_dir = f'/content/LarNO/exp/{latest_exp}/visualization'
    if os.path.exists(vis_dir):
        for root, dirs, files in os.walk(vis_dir):
            for f in files:
                print(' ', os.path.join(root, f))

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Display a grid of PNG snapshots from the first event
png_files = sorted(glob.glob(
    f'/content/LarNO/exp/{latest_exp}/visualization/region1_20m/**/*.png',
    recursive=True
))

if png_files:
    # Show up to 6 snapshots evenly spaced
    n_show = min(6, len(png_files))
    indices = [int(i * (len(png_files) - 1) / (n_show - 1)) for i in range(n_show)]
    selected = [png_files[i] for i in indices]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('LarNO vs MIKE+ — Flood depth snapshots (region1_20m)', fontsize=13)
    for ax, path in zip(axes.flat, selected):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Showing {n_show} of {len(png_files)} snapshots.')
else:
    print('No PNG snapshots found.')

In [ ]:
# (Optional) Package and download all results
import os

OUTPUT_ZIP = f'/content/LarNO_results_{latest_exp}.zip'
!cd /content/LarNO && zip -r {OUTPUT_ZIP} exp/{latest_exp}/ -q
print(f'Results packaged: {OUTPUT_ZIP}')
print('Use the Colab file browser (left panel) to download it, or run the cell below.')

# Colab download link
from google.colab import files
files.download(OUTPUT_ZIP)

---
## Citation

If you use LarNO in your research, please cite:

```bibtex
@article{larno2025,
  title   = {Large-scale urban flood modeling and zero-shot high-resolution generalization with LarNO},
  author  = {[TODO: authors]},
  journal = {[TODO: journal]},
  year    = {2025},
  doi     = {[TODO: DOI]}
}
```

---
**Links:** [GitHub](https://github.com/holmescao/LarNO) | [Dataset](https://holmescao.github.io/datasets/LarNO) | [Pre-trained Weights](https://drive.google.com/file/d/1ITPoTWQkm5v9kdZT9fqza2Xd4a6Lc-0t/view?usp=drive_link)